In [1]:
# ═══════════════════════════════════════════════════════
# NEPALI NLP — Translation + Sentiment + Understanding
# Using HuggingFace Transformers
# ═══════════════════════════════════════════════════════

from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

print("Testing Nepali Translation...")

# Nepali to English translation
translator = pipeline(
    "translation",
    model="Helsinki-NLP/opus-mt-ne-en"  # Nepali to English
)

test_sentences = [
    "नेपाल एक सुन्दर देश हो",
    "काठमाडौं नेपालको राजधानी हो",
    "मलाई नेपाली खाना मन पर्छ",
    "आज मौसम राम्रो छ",
]

print("\nNepali → English Translation:")
print("─" * 50)
for sentence in test_sentences:
    result = translator(sentence)
    print(f"नेपाली: {sentence}")
    print(f"English: {result[0]['translation_text']}")
    print()

/home/kulraj/nepali_ocr/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Testing Nepali Translation...


OSError: Helsinki-NLP/opus-mt-ne-en is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [2]:
# ── Use correct model name ────────────────────────────────────
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

print("Loading Nepali translator...")

translator = pipeline(
    "translation",
    model="Helsinki-NLP/opus-mt-mul-en"  # multilingual to English
)

test_sentences = [
    "नेपाल एक सुन्दर देश हो",
    "काठमाडौं नेपालको राजधानी हो",
    "मलाई नेपाली खाना मन पर्छ",
]

print("\nNepali → English:")
print("─" * 50)
for sentence in test_sentences:
    result = translator(sentence)
    print(f"नेपाली: {sentence}")
    print(f"English: {result[0]['translation_text']}")
    print()

Loading Nepali translator...


KeyError: "Unknown task translation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [3]:
# ── Fix — use correct pipeline task ──────────────────────────
from transformers import MarianMTModel, MarianTokenizer

print("Loading Nepali → English model...")

model_name = "Helsinki-NLP/opus-mt-mul-en"
tokenizer  = MarianTokenizer.from_pretrained(model_name)
mt_model   = MarianMTModel.from_pretrained(model_name)

print("Model loaded!")

def translate_nepali(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True)
    translated = mt_model.generate(**inputs)
    return tokenizer.decode(translated[0], skip_special_tokens=True)

# Test
test_sentences = [
    "नेपाल एक सुन्दर देश हो",
    "काठमाडौं नेपालको राजधानी हो",
    "मलाई नेपाली खाना मन पर्छ",
]

print("\nNepali → English:")
print("─" * 50)
for sentence in test_sentences:
    translation = translate_nepali(sentence)
    print(f"नेपाली: {sentence}")
    print(f"English: {translation}")
    print()

Loading Nepali → English model...


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 258/258 [00:00<00:00, 31592.28it/s]


Model loaded!

Nepali → English:
──────────────────────────────────────────────────
नेपाली: नेपाल एक सुन्दर देश हो
English: Nepal is a beautiful country

नेपाली: काठमाडौं नेपालको राजधानी हो
English: Napal is the capital of Kakatmado

नेपाली: मलाई नेपाली खाना मन पर्छ
English: Malai Nepali wants to eat.



In [4]:
# ── Keyword extraction + basic sentiment ─────────────────────
def analyze_nepali(text):
    # Translation
    translation = translate_nepali(text)
    
    # Basic keyword extraction — nouns and important words
    # Remove common Nepali stop words
    stop_words = ['एक', 'को', 'मा', 'छ', 'हो', 'र', 'मलाई', 
                  'तपाईं', 'यो', 'त्यो', 'गर्न', 'भयो', 'छन्',
                  'भने', 'गरे', 'हुन्छ', 'पर्छ', 'हुन्']
    
    words    = text.split()
    keywords = [w for w in words if w not in stop_words and len(w) > 1]
    
    # Basic sentiment — positive/negative word lists
    positive_words = ['राम्रो', 'सुन्दर', 'राम्री', 'खुशी', 'आनन्द', 
                      'प्रेम', 'मन', 'शुभ', 'सफल', 'उत्कृष्ट']
    negative_words = ['नराम्रो', 'दुःख', 'समस्या', 'गाह्रो', 'कठिन',
                      'रोग', 'मृत्यु', 'हार', 'असफल', 'खराब']
    
    pos_count = sum(1 for w in words if w in positive_words)
    neg_count = sum(1 for w in words if w in negative_words)
    
    if pos_count > neg_count:
        sentiment = "सकारात्मक (Positive) 😊"
    elif neg_count > pos_count:
        sentiment = "नकारात्मक (Negative) 😔"
    else:
        sentiment = "तटस्थ (Neutral) 😐"
    
    return {
        'original':    text,
        'translation': translation,
        'keywords':    keywords,
        'sentiment':   sentiment,
        'word_count':  len(words)
    }

# Test
test_texts = [
    "नेपाल एक सुन्दर देश हो",
    "आज मौसम राम्रो छ र म खुशी छु",
    "यो समस्या धेरै गाह्रो छ",
]

for text in test_texts:
    result = analyze_nepali(text)
    print(f"Text:        {result['original']}")
    print(f"Translation: {result['translation']}")
    print(f"Keywords:    {result['keywords']}")
    print(f"Sentiment:   {result['sentiment']}")
    print(f"Word count:  {result['word_count']}")
    print("─" * 50)

Text:        नेपाल एक सुन्दर देश हो
Translation: Nepal is a beautiful country
Keywords:    ['नेपाल', 'सुन्दर', 'देश']
Sentiment:   सकारात्मक (Positive) 😊
Word count:  5
──────────────────────────────────────────────────
Text:        आज मौसम राम्रो छ र म खुशी छु
Translation: Today, Ramiro is six years old.
Keywords:    ['आज', 'मौसम', 'राम्रो', 'खुशी', 'छु']
Sentiment:   सकारात्मक (Positive) 😊
Word count:  8
──────────────────────────────────────────────────
Text:        यो समस्या धेरै गाह्रो छ
Translation: The problem is that you don't know what you're talking about.
Keywords:    ['समस्या', 'धेरै', 'गाह्रो']
Sentiment:   नकारात्मक (Negative) 😔
Word count:  5
──────────────────────────────────────────────────


In [5]:
# ── Complete OCR + NLP Pipeline ───────────────────────────────
import torch
import torch.nn as nn
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import torchvision.transforms as T

# Load CRNN model
class CRNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),  nn.BatchNorm2d(64),  nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1),nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, 3, padding=1),nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d((2,1)),
            nn.Conv2d(256, 512, 3, padding=1),nn.BatchNorm2d(512), nn.ReLU(),
            nn.MaxPool2d((2,1)),
            nn.Conv2d(512, 512, 3, padding=1),nn.BatchNorm2d(512), nn.ReLU(),
            nn.MaxPool2d((2,1)),
        )
        self.rnn = nn.LSTM(512*2, 256, num_layers=2,
                           bidirectional=True, batch_first=True, dropout=0.3)
        self.fc  = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        b, c, h, w = x.size()
        x = x.permute(0, 3, 1, 2).reshape(b, w, c*h)
        x, _ = self.rnn(x)
        return self.fc(x).log_softmax(2)

CHARS = ['<blank>',
         'क','ख','ग','घ','ङ','च','छ','ज','झ','ञ',
         'ट','ठ','ड','ढ','ण','त','थ','द','ध','न',
         'प','फ','ब','भ','म','य','र','ल','व','श',
         'ष','स','ह','क्ष','त्र','ज्ञ',
         '०','१','२','३','४','५','६','७','८','९',
         'ा','ि','ी','ु','ू','े','ै','ो','ौ','ं','ः','्','ँ','ृ',
         'अ','आ','इ','ई','उ','ऊ','ए','ऐ','ओ','औ','अं','अः']

idx2char  = {i: c for i, c in enumerate(CHARS)}
device    = torch.device('cpu')
crnn      = CRNN(num_classes=len(CHARS))
crnn.load_state_dict(torch.load('/home/kulraj/nepali_ocr/models/crnn_nepali_v4.pth',
                                 map_location=device))
crnn.eval()
print("CRNN loaded!")

FONT_PATH = '/usr/share/fonts/truetype/noto/NotoSerifDevanagari-Bold.ttf'

def ctc_decode(output):
    pred_ids = output.argmax(1).tolist()
    chars = []; prev = -1
    for idx in pred_ids:
        if idx != prev and idx != 0:
            chars.append(idx2char[idx])
        prev = idx
    return ''.join(chars)

def ocr_word(word):
    font   = ImageFont.truetype(FONT_PATH, size=48)
    img    = Image.new('L', (256, 64), color=255)
    draw   = ImageDraw.Draw(img)
    bbox   = draw.textbbox((0,0), word, font=font)
    w, h   = bbox[2]-bbox[0], bbox[3]-bbox[1]
    draw.text(((256-w)//2, (64-h)//2), word, font=font, fill=0)
    img_t  = T.ToTensor()(img)
    img_t  = T.Normalize((0.5,), (0.5,))(img_t)
    img_t  = img_t.unsqueeze(0)
    with torch.no_grad():
        out  = crnn(img_t)
        pred = ctc_decode(out[0])
    return pred

def full_pipeline(text):
    print(f"Input text:  {text}")
    print()
    
    # Step 1 — OCR each word
    words     = text.split()
    ocr_words = [ocr_word(w) for w in words]
    ocr_text  = ' '.join(ocr_words)
    print(f"OCR output:  {ocr_text}")
    
    # Step 2 — NLP analysis
    result = analyze_nepali(ocr_text)
    print(f"Translation: {result['translation']}")
    print(f"Keywords:    {result['keywords']}")
    print(f"Sentiment:   {result['sentiment']}")
    print(f"Word count:  {result['word_count']}")
    print("─" * 50)

# Test full pipeline
test_inputs = [
    "नेपाल सुन्दर देश",
    "आज राम्रो मौसम",
    "काठमाडौं राजधानी",
]

print("Complete OCR + NLP Pipeline:")
print("═" * 50)
for text in test_inputs:
    full_pipeline(text)

CRNN loaded!
Complete OCR + NLP Pipeline:
══════════════════════════════════════════════════
Input text:  नेपाल सुन्दर देश

OCR output:  नेपाल सुन्दर ढोरा
Translation: Nepalese Rupee
Keywords:    ['नेपाल', 'सुन्दर', 'ढोरा']
Sentiment:   सकारात्मक (Positive) 😊
Word count:  3
──────────────────────────────────────────────────
Input text:  आज राम्रो मौसम

OCR output:  आज राम्रो सकम
Translation: Today’s Ramiro is capable of
Keywords:    ['आज', 'राम्रो', 'सकम']
Sentiment:   सकारात्मक (Positive) 😊
Word count:  3
──────────────────────────────────────────────────
Input text:  काठमाडौं राजधानी

OCR output:  काठमाडौं राजदानी
Translation: KADAMADO King
Keywords:    ['काठमाडौं', 'राजदानी']
Sentiment:   तटस्थ (Neutral) 😐
Word count:  2
──────────────────────────────────────────────────


In [6]:
# ── Local Nepali Spell Corrector ──────────────────────────────
# Uses edit distance to find closest correct word

def edit_distance(s1, s2):
    m, n = len(s1), len(s2)
    dp   = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j],
                                   dp[i][j-1],
                                   dp[i-1][j-1])
    return dp[m][n]

# Nepali word dictionary
NEPALI_DICTIONARY = [
    # Places
    'नेपाल','काठमाडौं','पोखरा','जनकपुर','भक्तपुर',
    'ललितपुर','बिराटनगर','धरान','इटहरी','बुटवल',
    'नारायणगढ','हेटौडा','दमक','इलाम','तापलेजुङ',
    'राजधानी','सहर','गाउँ','नगर','जिल्ला',
    # Common words
    'नमस्ते','धन्यवाद','शुभकामना','स्वागत','बिदाई',
    'राम्रो','राम्री','सुन्दर','अग्लो','होचो',
    'पानी','खाना','घर','ढोका','झ्याल',
    'किताब','कलम','कापी','थैला','जुत्ता',
    'आमा','बुवा','दाई','दिदी','भाई','बहिनी',
    'साथी','शिक्षक','विद्यार्थी','विद्यालय',
    'सूर्य','चन्द्र','तारा','आकाश','पृथ्वी',
    'नदी','पहाड','जंगल','बाटो','खेत',
    'देश','समाज','सरकार','जनता','अधिकार',
    'एक','दुई','तीन','चार','पाँच',
    'छ','सात','आठ','नौ','दस',
    'आज','भोलि','हिजो','अहिले','पछि',
    'राम्रो','नराम्रो','ठूलो','सानो','धेरै',
    'मौसम','वर्षा','हिउँ','घाम','बादल',
    # Sentences words
    'एक','सुन्दर','हो','छ','र','को','मा',
    'गर्न','भयो','छन्','भने','हुन्छ',
]

def correct_ocr(text, max_distance=2):
    words         = text.split()
    corrected     = []
    corrections   = []

    for word in words:
        # Find closest word in dictionary
        best_word = word
        best_dist = max_distance + 1

        for dict_word in NEPALI_DICTIONARY:
            dist = edit_distance(word, dict_word)
            if dist < best_dist:
                best_dist = dist
                best_word = dict_word

        corrected.append(best_word)
        if best_word != word:
            corrections.append(f"{word} → {best_word}")

    return ' '.join(corrected), corrections

# Test
test_ocr_outputs = [
    "काठमाडौं राजदानी हो",      # राजदानी should be राजधानी
    "नेपाल सुन्दर दश हो",        # दश should be देश
    "आज मौसम रामो छ",           # रामो should be राम्रो
]

print("OCR Spell Correction:")
print("═" * 50)
for text in test_ocr_outputs:
    corrected, corrections = correct_ocr(text)
    print(f"Original:  {text}")
    print(f"Corrected: {corrected}")
    if corrections:
        print(f"Fixed:     {corrections}")
    else:
        print(f"Fixed:     No corrections needed")
    print("─" * 50)

OCR Spell Correction:
══════════════════════════════════════════════════
Original:  काठमाडौं राजदानी हो
Corrected: काठमाडौं राजधानी हो
Fixed:     ['राजदानी → राजधानी']
──────────────────────────────────────────────────
Original:  नेपाल सुन्दर दश हो
Corrected: नेपाल सुन्दर देश हो
Fixed:     ['दश → देश']
──────────────────────────────────────────────────
Original:  आज मौसम रामो छ
Corrected: आज मौसम राम्रो छ
Fixed:     ['रामो → राम्रो']
──────────────────────────────────────────────────


In [7]:
# ── Complete pipeline with spell correction ───────────────────
def full_pipeline_v2(ocr_text):
    print(f"Raw OCR:     {ocr_text}")
    
    # Step 1 — Spell correction
    corrected, fixes = correct_ocr(ocr_text)
    print(f"Corrected:   {corrected}")
    if fixes:
        print(f"Fixes made:  {fixes}")
    
    # Step 2 — Translation
    translation = translate_nepali(corrected)
    print(f"Translation: {translation}")
    
    # Step 3 — Sentiment
    sentiment = get_sentiment(corrected)
    print(f"Sentiment:   {sentiment}")
    
    # Step 4 — Keywords
    stop_words = ['एक','को','मा','छ','हो','र','मलाई',
                  'तपाईं','यो','त्यो','गर्न','भयो','छन्']
    keywords = [w for w in corrected.split() if w not in stop_words and len(w) > 1]
    print(f"Keywords:    {keywords}")
    print("─" * 50)

# Test
test_inputs = [
    "काठमाडौं राजदानी हो",
    "नेपाल सुन्दर दश हो",
    "आज मौसम रामो छ",
]

print("Complete Pipeline with Spell Correction:")
print("═" * 50)
for text in test_inputs:
    full_pipeline_v2(text)

Complete Pipeline with Spell Correction:
══════════════════════════════════════════════════
Raw OCR:     काठमाडौं राजदानी हो
Corrected:   काठमाडौं राजधानी हो
Fixes made:  ['राजदानी → राजधानी']
Translation: KADAMADO is a capitalist.


NameError: name 'get_sentiment' is not defined

In [8]:
# ── Define missing functions ──────────────────────────────────
def get_sentiment(text):
    positive_words = ['राम्रो','सुन्दर','राम्री','खुशी','आनन्द',
                      'प्रेम','मन','शुभ','सफल','उत्कृष्ट']
    negative_words = ['नराम्रो','दुःख','समस्या','गाह्रो','कठिन',
                      'रोग','मृत्यु','हार','असफल','खराब']
    words     = text.split()
    pos_count = sum(1 for w in words if w in positive_words)
    neg_count = sum(1 for w in words if w in negative_words)
    if pos_count > neg_count:
        return "सकारात्मक (Positive) 😊"
    elif neg_count > pos_count:
        return "नकारात्मक (Negative) 😔"
    return "तटस्थ (Neutral) 😐"

def get_keywords(text):
    stop_words = ['एक','को','मा','छ','हो','र','मलाई',
                  'तपाईं','यो','त्यो','गर्न','भयो','छन्',
                  'भने','गरे','हुन्छ','पर्छ','हुन्']
    return [w for w in text.split() if w not in stop_words and len(w) > 1]

# ── Test complete pipeline ────────────────────────────────────
test_inputs = [
    "काठमाडौं राजदानी हो",
    "नेपाल सुन्दर दश हो",
    "आज मौसम रामो छ",
]

print("Complete Pipeline with Spell Correction:")
print("═" * 50)
for text in test_inputs:
    corrected, fixes   = correct_ocr(text)
    translation        = translate_nepali(corrected)
    sentiment          = get_sentiment(corrected)
    keywords           = get_keywords(corrected)

    print(f"Raw OCR:     {text}")
    print(f"Corrected:   {corrected}")
    print(f"Fixes:       {fixes if fixes else 'None'}")
    print(f"Translation: {translation}")
    print(f"Sentiment:   {sentiment}")
    print(f"Keywords:    {keywords}")
    print("─" * 50)

Complete Pipeline with Spell Correction:
══════════════════════════════════════════════════
Raw OCR:     काठमाडौं राजदानी हो
Corrected:   काठमाडौं राजधानी हो
Fixes:       ['राजदानी → राजधानी']
Translation: KADAMADO is a capitalist.
Sentiment:   तटस्थ (Neutral) 😐
Keywords:    ['काठमाडौं', 'राजधानी']
──────────────────────────────────────────────────
Raw OCR:     नेपाल सुन्दर दश हो
Corrected:   नेपाल सुन्दर देश हो
Fixes:       ['दश → देश']
Translation: It's Nepal. It's a beautiful country.
Sentiment:   सकारात्मक (Positive) 😊
Keywords:    ['नेपाल', 'सुन्दर', 'देश']
──────────────────────────────────────────────────
Raw OCR:     आज मौसम रामो छ
Corrected:   आज मौसम राम्रो छ
Fixes:       ['रामो → राम्रो']
Translation: Today’s Six - Year - Old Ramiro
Sentiment:   सकारात्मक (Positive) 😊
Keywords:    ['आज', 'मौसम', 'राम्रो']
──────────────────────────────────────────────────
